# Rhea FinGraph — XGBoost Training on Kaggle T4

Trains both model variants on the free Tesla T4 GPU (zero MacBook heat):
1. **online** — cold-start-safe features; this serves the live API
2. **full** — history-aware benchmark variant

**Before running:** Session options → Accelerator → GPU T4, and Input must include your `rhea-fingraph-ibm-splits` dataset.
**After running:** File → Save & Run All (Commit), then download the two artifact folders from the Output tab into local `artifacts/models/`.

In [ ]:
%pip install -q -U xgboost polars
%pip install -q --force-reinstall --no-deps git+https://github.com/aditisahu1234/Rhea-FinGraph.git

In [ ]:
import glob

import torch

paths = {p.split("/")[-1]: p for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True)}
print("Found:", sorted(paths))
assert {"train.parquet", "validation.parquet", "test.parquet"} <= set(paths), paths
TRAIN, VAL, TEST = paths["train.parquet"], paths["validation.parquet"], paths["test.parquet"]

assert torch.cuda.is_available(), "Enable the GPU accelerator in Session options!"
print("GPU:", torch.cuda.get_device_name(0))

## 1) Train the ONLINE (serving) model

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.train_baseline",
        "--backend", "xgboost", "--device", "cuda", "--feature-set", "online",
        "--train", TRAIN, "--val", VAL, "--test", TEST,
        "--out", "/kaggle/working/baseline-online-xgb",
    ],
    check=True,
)

## 2) Train the FULL (benchmark) model

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.train_baseline",
        "--backend", "xgboost", "--device", "cuda", "--feature-set", "full",
        "--train", TRAIN, "--val", VAL, "--test", TEST,
        "--out", "/kaggle/working/baseline-full-xgb",
    ],
    check=True,
)

In [ ]:
!ls -lh /kaggle/working/baseline-online-xgb /kaggle/working/baseline-full-xgb
print("\nDone. Save Version -> download both folders from the Output tab.")